In [1]:
import numpy as np
from jax import jit
from jax import lax
from jax import random
import jax
import jax.numpy as jnp

In [2]:
def impure_print_side_effect(x):
  print("Executing function")  # This is a side-effect
  return x

# The side-effects appear during the first run
print ("First call: ", jit(impure_print_side_effect)(4.))

# Subsequent runs with parameters of same type and shape may not show the side-effect
# This is because JAX now invokes a cached compilation of the function
print ("Second call: ", jit(impure_print_side_effect)(5.))

# JAX re-runs the Python function when the type or shape of the argument changes
print ("Third call, different type: ", jit(impure_print_side_effect)(jnp.array([5.])))

Executing function
First call:  4.0
Second call:  5.0
Executing function
Third call, different type:  [5.]


In [3]:
from jax import make_jaxpr

array = jnp.arange(10)

In [4]:
array

Array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32)

See [`lax.fori_loop`](https://docs.jax.dev/en/latest/_autosummary/jax.lax.fori_loop.html#jax.lax.fori_loop)

In [5]:
lax.fori_loop(0, 10, lambda i, x: x+array[i], 0)

Array(45, dtype=int32)

In [6]:
sum(array)

Array(45, dtype=int32)

In [7]:
def foo(i, x):
    result = x+array[i]
    jax.debug.print("i={i}, x={x}, result={result}", i=i, x=x, result=result)
    return result

In [8]:
lax.fori_loop(0, 10, foo, 0)

i=0, x=0, result=0
i=1, x=0, result=1
i=2, x=1, result=3
i=3, x=3, result=6
i=4, x=6, result=10
i=5, x=10, result=15
i=6, x=15, result=21
i=7, x=21, result=28
i=8, x=28, result=36
i=9, x=36, result=45


Array(45, dtype=int32)

In [9]:
jax_array = jnp.zeros((3,3), dtype=jnp.float32)

# In place update of JAX's array will yield an error!
try:
    jax_array[1, :] = 1.0
except Exception as ex:
    print("Exception", ex)

Exception JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html


In [10]:
class Foo:
    def __init__(self, x):
        self.x = x

In [11]:
foo = Foo(1)

In [12]:
def bar(f):
    return f.x + 3

In [13]:
bar(foo)

4

In [14]:
@jit
def bar(f):
    return f.x + 3

In [15]:
try:
    bar(foo)
except Exception as ex:
    print("Exception", ex)

Exception Error interpreting argument to <function bar at 0x7f8304168e00> as an abstract array. The problematic value is of type <class '__main__.Foo'> and was passed to the function at path f.
This typically means that a jit-wrapped function was called with a non-array argument, and this argument was not marked as static using the static_argnums or static_argnames parameters of jax.jit.


In [16]:
from functools import partial

@partial(jit, static_argnums=0)
def bar(f):
    return f.x + 3

In [17]:
bar(foo)

Array(4, dtype=int32, weak_type=True)

In [18]:
foo.x += 1

In [19]:
bar(foo)

Array(4, dtype=int32, weak_type=True)

In [20]:
class Foo:
    def __init__(self, x):
        self.x = x

    def _tree_flatten(self):
        children = (self.x,)
        aux_data = {}
        return (children, aux_data)

    @classmethod
    def _tree_unflatten(cls, aux_data, children):
        return cls(*children)

from jax import tree_util
tree_util.register_pytree_node(Foo, Foo._tree_flatten, Foo._tree_unflatten)

In [25]:
@jit
def bar(f):
    return f.x + 3

In [26]:
foo = Foo(1)

In [27]:
bar(foo)

Array(4, dtype=int32, weak_type=True)

In [28]:
foo.x += 1

In [29]:
bar(foo)

Array(5, dtype=int32, weak_type=True)